**On commence par lire les fichers et les vectorisés**

In [4]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, TextLoader

# >>> Si tes fichiers sont des PDF :
loader = DirectoryLoader("documents/", glob="*.pdf", loader_cls=PyPDFLoader)

# >>> Si tes fichiers sont des .txt, commente la ligne du dessus
#     et décommente celle-ci à la place :
# loader = DirectoryLoader("documents/", glob="*.txt", loader_cls=TextLoader)

documents = loader.load()
print(f"Nombre de documents/pages chargés : {len(documents)}")
print("Aperçu du début du 1er :", documents[0].page_content[:200])

Nombre de documents/pages chargés : 91
Aperçu du début du 1er : Intelligence artificielle
Partie de Informatique, nouvelles
technologies, raisonnement
Pratiqué parChercheur ou chercheuse en
intelligence artiﬁcielle (d),
ingénieur en intelligence
artiﬁcielle
Champs


In [5]:
#decouper les documents en chunk (en morceaaux)
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=5000, chunk_overlap=500)
chunks = splitter.split_documents(documents)

#afficher pour voir le nombre de morceaux crée
print(f"Nombre de morceaux créées : {len(chunks)}")
print("Exemple de morceau : ", chunks[0].page_content[:200])


Nombre de morceaux créées : 103
Exemple de morceau :  Intelligence artificielle
Partie de Informatique, nouvelles
technologies, raisonnement
Pratiqué parChercheur ou chercheuse en
intelligence artiﬁcielle (d),
ingénieur en intelligence
artiﬁcielle
Champs


In [6]:
import time
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

taille_lot = 40   # bien en dessous de la limite de 100/min
pause = 60        # pause d'1 minute entre les lots

# Premier lot : on crée la base
vectorstore = FAISS.from_documents(chunks[:taille_lot], embeddings)
print(f"Lot 1 traité ({len(chunks[:taille_lot])} morceaux)")

# Lots suivants : on attend, puis on ajoute
for debut in range(taille_lot, len(chunks), taille_lot):
    print(f"Pause de {pause}s pour respecter la limite gratuite...")
    time.sleep(pause)
    lot = chunks[debut:debut + taille_lot]
    vectorstore.add_documents(lot)
    print(f"Lot ajouté ({len(lot)} morceaux)")

vectorstore.save_local("faiss_index")
print("Base vectorielle créée et sauvegardée !")

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

In [ ]:
#là maintenant faire un peu du sroll levellll